# REAL‑E‑CON — Knowledge Tree (Graphically‑Stunning Book Visualization)

This notebook renders the **book as a Tree of Knowledge** using your Neo4j graph.

It loads the hierarchy from Neo4j as:

- `(:Book)-[:HAS_OUTLINE]->(:Outline)`
- `(:Outline)-[:HAS_CHILD]->(:Outline)`
- `(:Outline)-[:HAS_CONCEPT]->(:Concept)` (materialized by your ingestion v2)

If you have multiple books, you can set `REAL_E_CON_SELECT_BOOK_ID` as an environment variable to select one; otherwise the first `:Book` is used.


## 0) Setup

In [6]:
# If needed (uncomment):
# %pip -q install plotly pyvis neo4j pandas networkx ipywidgets

import os
import json
from typing import Optional, Tuple

import pandas as pd
import networkx as nx

import plotly.express as px
from IPython.display import HTML, display

try:
    from neo4j import GraphDatabase
    _HAS_NEO4J = True
except Exception:
    _HAS_NEO4J = False

try:
    from pyvis.network import Network
    _HAS_PYVIS = True
except Exception:
    _HAS_PYVIS = False

print("Neo4j driver:", "OK" if _HAS_NEO4J else "NOT installed")
print("PyVis:", "OK" if _HAS_PYVIS else "NOT installed")


Neo4j driver: OK
PyVis: OK


## 1) Config

In [7]:
NEO4J_URI      = os.getenv("NEO4J_URI", "neo4j://localhost:7687")
NEO4J_USER     = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "testpassword")
NEO4J_PASS = os.getenv("NEO4J_PASS", os.getenv("NEO4J_PASSWORD", "testpassword"))

OUTLINE_CSV  = os.getenv("REAL_E_CON_OUTLINE_CSV", "outline.csv")
OUTLINE_JSON = os.getenv("REAL_E_CON_OUTLINE_JSON", "outline.json")

BOOK_TITLE = os.getenv("REAL_E_CON_BOOK_TITLE", "Microeconomics")

MAX_CONCEPTS   = int(os.getenv("REAL_E_CON_MAX_CONCEPTS", "8000"))
MAX_XREF_EDGES = int(os.getenv("REAL_E_CON_MAX_XREF_EDGES", "12000"))

# Optional: set this to a specific book_id if you have multiple books; otherwise first book is used.
SELECT_BOOK_ID = os.getenv('REAL_E_CON_SELECT_BOOK_ID', '').strip() or None

# If True, do NOT fall back to demo/CSV/JSON when Neo4j fails; raise loudly.
FORCE_NEO4J = True


In [10]:
# ================================
# Neo4j sanity check (must be > 9 nodes)
# ================================
if not _HAS_NEO4J:
    raise RuntimeError("neo4j driver not installed. Install neo4j Python package.")

drv = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
with drv.session() as sess:
    books = sess.run("MATCH (b:Book) RETURN count(b) AS n").single()["n"]
    if books == 0:
        raise RuntimeError("No :Book nodes found. Likely wrong NEO4J_URI/DB.")
    b = sess.run(
        "MATCH (b:Book) "
        "RETURN b.book_id AS book_id, coalesce(b.title,b.name,'Book') AS title "
        "ORDER BY b.book_id LIMIT 1"
    ).single()
    book_id = b["book_id"]
    counts = sess.run(
        """
        MATCH (b:Book {book_id:$book_id})
        OPTIONAL MATCH (b)-[:HAS_OUTLINE]->(o:Outline)
        OPTIONAL MATCH (o)-[hc:HAS_CHILD]->(:Outline)
        OPTIONAL MATCH (o)-[hcon:HAS_CONCEPT]->(:Concept)
        RETURN
        count(DISTINCT o) AS outlines,
        count(DISTINCT hc) AS has_child_edges,
        count(DISTINCT hcon) AS has_concept_edges
        """,
        book_id=book_id
    ).single()
drv.close()

print("Neo4j OK")
print("books:", books)
print("using book_id:", book_id)
print("outlines:", counts["outlines"])
print("HAS_CHILD edges:", counts["has_child_edges"])
print("HAS_CONCEPT edges:", counts["has_concept_edges"])
if counts["outlines"] < 10:
    raise RuntimeError("Outlines < 10. This indicates wrong DB or ingestion did not run.")


Neo4j OK
books: 1
using book_id: pindyck_micro_9e
outlines: 17
HAS_CHILD edges: 19
HAS_CONCEPT edges: 17


## 2) Load hierarchy

In [11]:
def _safe_read_outline_csv(path: str) -> Optional[pd.DataFrame]:
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path)
    needed = {"id", "parent", "label"}
    if not needed.issubset(set(df.columns)):
        raise ValueError(f"CSV must include at least columns: {sorted(needed)}. Found: {list(df.columns)}")
    if "type" not in df.columns:  df["type"] = "node"
    if "value" not in df.columns: df["value"] = 1
    if "pages" not in df.columns: df["pages"] = None
    return df

def _safe_read_outline_json(path: str) -> Optional[pd.DataFrame]:
    if not os.path.exists(path):
        return None
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows = []
    def walk(node, parent=None, counter=[0]):
        counter[0] += 1
        nid = node.get("id") or f"n{counter[0]}"
        label = node.get("label") or node.get("name") or node.get("title") or str(nid)
        ntype = node.get("type") or ("root" if parent is None else "node")
        value = node.get("value", 1)
        pages = node.get("pages", None)
        rows.append({"id": nid, "parent": parent or "", "label": label, "type": ntype, "value": value, "pages": pages})
        for ch in node.get("children", []) or []:
            walk(ch, nid, counter)

    if isinstance(data, list):
        for item in data:
            rows.append({
                "id": item["id"],
                "parent": item.get("parent",""),
                "label": item.get("label") or item.get("name") or item.get("title"),
                "type": item.get("type","node"),
                "value": item.get("value",1),
                "pages": item.get("pages",None),
            })
    elif isinstance(data, dict):
        walk(data, None)
    else:
        raise ValueError("Unsupported JSON format for outline")

    df = pd.DataFrame(rows)
    if "value" not in df.columns: df["value"] = 1
    if "pages" not in df.columns: df["pages"] = None
    return df

def _demo_outline() -> pd.DataFrame:
    nodes = [
        ("book","", BOOK_TITLE, "book", 1, None),
        ("ch1","book","Chapter 1 — Scarcity & Choice","chapter", 1, "1–30"),
        ("s11","ch1","1.1 Opportunity Cost","section", 1, "12–17"),
        ("c1","s11","Opportunity Cost","concept", 1, 12),
        ("s12","ch1","1.2 PPF","section", 1, "18–25"),
        ("c2","s12","Production Possibility Frontier (PPF)","concept", 1, 18),
        ("ch2","book","Chapter 2 — Markets & Prices","chapter", 1, "31–90"),
        ("s21","ch2","2.1 Demand","section", 1, "45–63"),
        ("c3","s21","Demand","concept", 1, 45),
    ]
    return pd.DataFrame(nodes, columns=["id","parent","label","type","value","pages"])

def load_outline() -> Tuple[pd.DataFrame, str]:
    """Load a proper Book→Outline→Concept hierarchy from Neo4j.
    Falls back to outline.csv / outline.json / demo if Neo4j is unavailable.
    """
    if _HAS_NEO4J:
        try:
            drv = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
            with drv.session() as sess:

                # If multiple books exist, default to the first; allow override via SELECT_BOOK_ID env var.
                book = sess.run("""
                MATCH (b:Book)
                WHERE ($sel IS NULL OR b.book_id = $sel)
                RETURN b.book_id AS book_id, coalesce(b.title, b.name, 'Book') AS title
                ORDER BY b.book_id
                LIMIT 1
                """, sel=SELECT_BOOK_ID).single()

                if not book:
                    raise RuntimeError("No :Book found in Neo4j (or SELECT_BOOK_ID did not match).")

                book_id = book["book_id"]
                book_title = book["title"] or BOOK_TITLE

                # Pull Outline hierarchy + attached Concepts. This avoids 'Chapter (unknown)' entirely.
                q = """
                // Outline -> Outline edges
                MATCH (b:Book {book_id:$book_id})-[:HAS_OUTLINE]->(o:Outline)
                OPTIONAL MATCH (o)-[:HAS_CHILD]->(c:Outline)
                RETURN
                  'child' AS edge_kind,
                  elementId(o) AS parent_eid,
                  coalesce(o.title, o.name, '(outline)') AS parent_label,
                  coalesce(o.kind, 'outline') AS parent_kind,
                  o.level AS parent_level, o.start_page AS parent_sp, o.end_page AS parent_ep,

                  elementId(c) AS child_eid,
                  coalesce(c.title, c.name, '(outline)') AS child_label,
                  coalesce(c.kind, 'outline') AS child_kind,
                  c.level AS child_level, c.start_page AS child_sp, c.end_page AS child_ep,

                  NULL AS concept_page

                UNION ALL

                // Outline -> Concept edges
                MATCH (b:Book {book_id:$book_id})-[:HAS_OUTLINE]->(o:Outline)-[:HAS_CONCEPT]->(k:Concept)
                RETURN
                  'concept' AS edge_kind,
                  elementId(o) AS parent_eid,
                  coalesce(o.title, o.name, '(outline)') AS parent_label,
                  coalesce(o.kind, 'outline') AS parent_kind,
                  o.level AS parent_level, o.start_page AS parent_sp, o.end_page AS parent_ep,

                  elementId(k) AS child_eid,
                  coalesce(k.name, k.title, '(concept)') AS child_label,
                  'concept' AS child_kind,
                  NULL AS child_level, k.start_page AS child_sp, k.end_page AS child_ep,

                  coalesce(k.first_page, k.start_page) AS concept_page
                """

                res = sess.run(q, book_id=book_id).data()
                if not res:
                    raise RuntimeError("No data returned. Check HAS_OUTLINE/HAS_CHILD/HAS_CONCEPT relationships.")

                def fmt_pages(sp, ep):
                    if sp is None and ep is None:
                        return None
                    if sp is None:
                        return f"–{ep}"
                    if ep is None:
                        return f"{sp}–"
                    return f"{sp}–{ep}"

                rows = [{
                    "id": "book",
                    "parent": "",
                    "label": book_title,
                    "type": "book",
                    "value": 1,
                    "pages": None
                }]
                seen = {"book"}
                edges = []

                def out_id(eid): return f"out:{eid}"
                def con_id(eid): return f"concept:{eid}"

                # Build nodes + edges
                for r in res:
                    p_eid = r["parent_eid"]
                    if p_eid is None:
                        continue
                    pid = out_id(p_eid)

                    if pid not in seen:
                        rows.append({
                            "id": pid,
                            "parent": "",  # filled later
                            "label": r["parent_label"],
                            "type": str(r.get("parent_kind") or "outline"),
                            "value": 1,
                            "pages": fmt_pages(r.get("parent_sp"), r.get("parent_ep")),
                        })
                        seen.add(pid)

                    if r["edge_kind"] == "child" and r.get("child_eid"):
                        c_eid = r["child_eid"]
                        cid = out_id(c_eid)
                        if cid not in seen:
                            rows.append({
                                "id": cid,
                                "parent": "",
                                "label": r["child_label"],
                                "type": str(r.get("child_kind") or "outline"),
                                "value": 1,
                                "pages": fmt_pages(r.get("child_sp"), r.get("child_ep")),
                            })
                            seen.add(cid)
                        edges.append((pid, cid))

                    if r["edge_kind"] == "concept" and r.get("child_eid"):
                        k_eid = r["child_eid"]
                        kid = con_id(k_eid)
                        if kid not in seen:
                            rows.append({
                                "id": kid,
                                "parent": "",
                                "label": r["child_label"],
                                "type": "concept",
                                "value": 1,
                                "pages": r.get("concept_page"),
                            })
                            seen.add(kid)
                        edges.append((pid, kid))

                # Outline roots: outlines that never appear as a child outline
                child_outlines = {v for u, v in edges if v.startswith("out:")}
                outline_nodes = [n for n in seen if n.startswith("out:")]
                root_outlines = [n for n in outline_nodes if n not in child_outlines]

                # Parent maps
                outline_parent = {v: u for (u, v) in edges if v.startswith("out:")}
                for ro in root_outlines:
                    outline_parent.setdefault(ro, "book")

                concept_parent = {v: u for (u, v) in edges if v.startswith("concept:")}

                for rr in rows:
                    rid = rr["id"]
                    if rid == "book":
                        continue
                    if rid.startswith("out:"):
                        rr["parent"] = outline_parent.get(rid, "book")
                    else:
                        rr["parent"] = concept_parent.get(rid, "book")

                df = pd.DataFrame(rows).drop_duplicates(subset=["id"])
                df["parent"] = df["parent"].fillna("").astype(str)
                if len(df) < 10 and ('FORCE_NEO4J' in globals() and FORCE_NEO4J):
                    raise RuntimeError(f"Neo4j returned only {len(df)} nodes. This indicates wrong DB or fallback.")
                return df, f"Neo4j v2 (Book→Outline→Concept) @ {NEO4J_URI} | book_id={book_id}"

        except Exception as e:
            print("Neo4j ingest failed; falling back. Error:", repr(e))
            if 'FORCE_NEO4J' in globals() and FORCE_NEO4J:
                raise

    df = _safe_read_outline_csv(OUTLINE_CSV)
    if df is not None:
        return df, f"CSV: {OUTLINE_CSV}"

    df = _safe_read_outline_json(OUTLINE_JSON)
    if df is not None:
        return df, f"JSON: {OUTLINE_JSON}"

    return _demo_outline(), "Demo"

df_nodes, source_used = load_outline()
print("Source:", source_used)
print("Nodes:", len(df_nodes))
df_nodes.head(10)


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `first_page` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=35, column=30, offset=1637>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 1637, 'line': 35, 'column': 30}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n                // Outline -> Outline edges\n                MATCH (b:Book {book_id:$book_id})-[:HAS_OUTLINE]->(o:Outline)\n                OPTIONAL MATCH (o)-[:HAS_CHILD]->(c:Outline)\n                RETURN\n                  'child' AS edge_kind,\n                  elementId(o) AS parent_eid,\n                  coalesce

Source: Neo4j v2 (Book→Outline→Concept) @ neo4j://localhost:7687 | book_id=pindyck_micro_9e
Nodes: 54


,id,parent,label,type,value,pages
0,book,,Microeconomics,book,1,None
1,out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:164,book,Copyright Page,chapter,1,6–7
2,out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:626,book,Answers to Selected Exercises,chapter,1,755–769
3,out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:629,book,List of Examples,chapter,1,785–787
4,out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:169,book,Part One Introduction: Markets and Prices,chapter,1,25–88
5,out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:187,out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:169,2 The Basics of Supply and Demand,subchapter,1,45–88
6,out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:170,out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:169,1 Preliminaries,subchapter,1,27–44
7,out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:163,book,Title Page,chapter,1,5–5
8,out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:165,book,About the Authors,chapter,1,8–8
9,out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:628,book,Index,chapter,1,771–784


## 3) Diagnostics (why did it collapse?)

In [12]:
df_dbg = df_nodes.copy()
df_dbg["id"] = df_dbg["id"].astype(str)
df_dbg["parent"] = df_dbg["parent"].fillna("").astype(str)

ids = set(df_dbg["id"])
roots = df_dbg[df_dbg["parent"].isin(["", "None", "nan"])]
orphans = df_dbg[(~df_dbg["parent"].isin(["", "None", "nan"])) & (~df_dbg["parent"].isin(ids))]

print("Total nodes:", len(df_dbg))
print("Root nodes:", len(roots))
print("Orphans (parent missing):", len(orphans))
print("Rows whose parent exists in ids:", int((df_dbg['parent'].isin(ids) & ~df_dbg['parent'].isin(['','None','nan'])).sum()))

if len(orphans):
    display(orphans.head(20))

display(df_dbg["parent"].value_counts().head(20))


Total nodes: 54
Root nodes: 1
Orphans (parent missing): 0
Rows whose parent exists in ids: 53


parent
book                                              17
out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:203     8
out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:385     7
out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:536     5
out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:169     3
                                                   1
out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:162     1
out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:168     1
out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:625     1
out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:627     1
out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:166     1
out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:624     1
out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:163     1
out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:628     1
out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:165     1
out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:629     1
out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:626     1
out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:164     1
out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c

## 4) Clean + compute depth / weights

In [13]:
df = df_nodes.copy()
df["id"] = df["id"].astype(str)
df["parent"] = df["parent"].fillna("").astype(str)
df["label"] = df["label"].astype(str)

if "type" not in df.columns:  df["type"] = "node"
if "value" not in df.columns: df["value"] = 1
if "pages" not in df.columns: df["pages"] = None

# Ensure exactly one root
roots = df[df["parent"].isin(["", "None", "nan"])]
if roots.empty:
    df = pd.concat([pd.DataFrame([{"id":"book","parent":"","label":BOOK_TITLE,"type":"book","value":1,"pages":None}]), df], ignore_index=True)
elif len(roots) > 1:
    super_root = "BOOK_ROOT"
    df.loc[df["parent"].isin(["", "None", "nan"]), "parent"] = super_root
    df = pd.concat([pd.DataFrame([{"id":super_root,"parent":"","label":BOOK_TITLE,"type":"book","value":1,"pages":None}]), df], ignore_index=True)

df = df[df["id"] != df["parent"]].copy()

parent_map = dict(zip(df["id"], df["parent"]))
children = df.groupby("parent")["id"].apply(list).to_dict()

def depth(nid: str) -> int:
    d = 0
    seen = set()
    while True:
        p = parent_map.get(nid, "")
        if p in ("", None, "None", "nan"):
            return d
        if p in seen:
            return d
        seen.add(p)
        nid = p
        d += 1

def leaf_count(nid: str) -> int:
    stack = [nid]
    seen = set()
    leaves = 0
    while stack:
        x = stack.pop()
        if x in seen:
            continue
        seen.add(x)
        kids = children.get(x, [])
        if not kids:
            leaves += 1
        else:
            stack.extend(kids)
    return max(leaves, 1)

df["depth"] = df["id"].apply(depth)
df["leaf_value"] = df["id"].apply(leaf_count)
df["viz_value"] = df["leaf_value"].clip(upper=800)

print("Depth range:", int(df["depth"].min()), "→", int(df["depth"].max()))
df.head(10)


Depth range: 0 → 2


,id,parent,label,type,value,pages,depth,leaf_value,viz_value
0,book,,Microeconomics,book,1,None,0,36,36
1,out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:164,book,Copyright Page,chapter,1,6–7,1,1,1
2,out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:626,book,Answers to Selected Exercises,chapter,1,755–769,1,1,1
3,out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:629,book,List of Examples,chapter,1,785–787,1,1,1
4,out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:169,book,Part One Introduction: Markets and Prices,chapter,1,25–88,1,3,3
5,out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:187,out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:169,2 The Basics of Supply and Demand,subchapter,1,45–88,2,1,1
6,out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:170,out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:169,1 Preliminaries,subchapter,1,27–44,2,1,1
7,out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:163,book,Title Page,chapter,1,5–5,1,1,1
8,out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:165,book,About the Authors,chapter,1,8–8,1,1,1
9,out:4:c6ad85a7-60db-4c3d-95b2-ec940b49c59e:628,book,Index,chapter,1,771–784,1,1,1


## 5) Radial Tree (Sunburst)

In [14]:
fig = px.sunburst(
    df,
    names="label",
    ids="id",
    parents="parent",
    values="viz_value",
    color="depth",
    color_continuous_scale="Turbo",
    hover_data={"type": True, "depth": True, "viz_value": True, "id": True, "parent": True, "pages": True},
)

fig.update_layout(
    title=f"REAL‑E‑CON — Tree of Knowledge (Source: {source_used})",
    margin=dict(t=60, l=10, r=10, b=10),
    height=900,
)

fig.update_traces(insidetextorientation="radial", textinfo="label", marker=dict(line=dict(width=0.6)))
fig.show()


## 6) Treemap

In [15]:
fig2 = px.treemap(
    df,
    names="label",
    ids="id",
    parents="parent",
    values="viz_value",
    color="depth",
    color_continuous_scale="Viridis",
    hover_data={"type": True, "depth": True, "viz_value": True, "id": True, "pages": True},
)

fig2.update_layout(title="Knowledge Treemap (zoom / drill-down)", margin=dict(t=60, l=10, r=10, b=10), height=900)
fig2.show()


## 7) Quick Neo4j checks

Run these in Neo4j Browser if it still looks wrong:

```cypher
CALL db.labels();
```

```cypher
MATCH (c:Concept) RETURN count(c) AS concepts;
```

If your concept nodes do not use `chapter/section` property names, edit the **coalesce(...)** list in the Neo4j query in Section 2.


In [18]:
# ==========================================================
# FIXED v2: Mind-Map Tree DIRECTLY from Neo4j (Book → Outline tree)
# - adds Book -> root Outline edges (HAS_OUTLINE) into the layout graph
# - removes the "everything on one line" bug
# ==========================================================
import os
import math
from collections import defaultdict, deque

import pandas as pd
import plotly.graph_objects as go
from neo4j import GraphDatabase

BOOK_ID = "pindyck_micro_9e"
MAX_DEPTH = 20                 # outline-depth only (literal injected)
MAX_OUTLINE_EDGES = 200000     # safety cap

# Visual knobs
RADIUS_STEP = 1.35
EDGE_ALPHA = 0.22
FIG_HEIGHT = 950
LABEL_DEPTH = 2                # show text only up to this outline depth (hover always)

# -----------------------------
# Credentials
# -----------------------------
neo4j_pass = (
    globals().get("NEO4J_PASS")
    or globals().get("NEO4J_PASSWORD")
    or os.getenv("NEO4J_PASS")
    or os.getenv("NEO4J_PASSWORD")
)
if not neo4j_pass:
    raise RuntimeError("Missing Neo4j password. Define NEO4J_PASS/NEO4J_PASSWORD or env var.")

drv = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, neo4j_pass))

depth = int(MAX_DEPTH)

# -----------------------------
# 1) Pull:
#   - book node
#   - root outlines (HAS_OUTLINE)
#   - reachable outlines (HAS_CHILD*0..depth)
#   - pruned HAS_CHILD edges among reachable outlines
# -----------------------------
cypher = f"""
MATCH (b:Book {{book_id:$book_id}})
OPTIONAL MATCH (b)-[:HAS_OUTLINE]->(root:Outline)
OPTIONAL MATCH (root)-[:HAS_CHILD*0..{depth}]->(o:Outline)
WITH b, collect(DISTINCT root) AS roots, (collect(DISTINCT root) + collect(DISTINCT o)) AS outs
UNWIND outs AS n
WITH b, roots, collect(DISTINCT n) AS nodes
OPTIONAL MATCH (a:Outline)-[r:HAS_CHILD]->(c:Outline)
WHERE a IN nodes AND c IN nodes
RETURN
  elementId(b) AS book_eid,
  coalesce(b.title,b.name,b.book_id) AS book_title,
  [x IN roots | elementId(x)] AS root_eids,
  [x IN nodes | {{
    eid: elementId(x),
    title: coalesce(x.title,x.name,x.outline_id),
    kind: coalesce(x.kind,'outline'),
    level: x.level,
    sp: x.start_page,
    ep: x.end_page
  }}] AS outline_nodes,
  collect({{src: elementId(a), dst: elementId(c)}}) AS child_edges
LIMIT 1
"""

with drv.session() as sess:
    rec = sess.run(cypher, book_id=BOOK_ID).single()
drv.close()

if rec is None:
    raise RuntimeError("Neo4j returned no record. Check NEO4J_URI/auth/BOOK_ID.")

book_eid = str(rec["book_eid"])
book_title = rec["book_title"]
root_eids = [str(x) for x in (rec["root_eids"] or [])]
outline_nodes = rec["outline_nodes"] or []
child_edges = rec["child_edges"] or []

if not outline_nodes:
    raise RuntimeError("No Outline nodes reachable from the book. Check HAS_OUTLINE/HAS_CHILD ingestion.")

print("Book:", BOOK_ID, "| title:", book_title)
print("Reachable outlines:", len(outline_nodes))
print("HAS_CHILD edges (pruned):", len(child_edges))
print("Root outlines (HAS_OUTLINE):", len(root_eids))

# -----------------------------
# 2) Build node tables
# -----------------------------
def fmt_pages(sp, ep):
    if sp is None and ep is None:
        return None
    if sp is None:
        return f"–{ep}"
    if ep is None:
        return f"{sp}–"
    return f"{sp}–{ep}"

nodes = []
nodes.append({"id": book_eid, "parent": "", "label": book_title, "type": "book", "pages": None})

for n in outline_nodes:
    nid = str(n["eid"])
    nodes.append({
        "id": nid,
        "parent": None,  # set below
        "label": (n["title"] if n["title"] is not None else nid),
        "type": "outline",
        "pages": fmt_pages(n.get("sp"), n.get("ep"))
    })

df = pd.DataFrame(nodes).drop_duplicates(subset=["id"]).reset_index(drop=True)
df["id"] = df["id"].astype(str)
dfi = df.set_index("id", drop=False)

outline_ids = set(df[df["type"] == "outline"]["id"])

# -----------------------------
# 3) Build parent/children map INCLUDING Book->root edges  (critical fix)
# -----------------------------
children = defaultdict(list)
parents = {}  # child -> parent

# (A) Book -> roots (HAS_OUTLINE)
for r in root_eids:
    if r in outline_ids:
        parents.setdefault(r, book_eid)
        children[book_eid].append(r)

# (B) Outline -> Outline (HAS_CHILD)
for e in child_edges:
    s = str(e["src"]); t = str(e["dst"])
    if s in outline_ids and t in outline_ids:
        children[s].append(t)
        parents.setdefault(t, s)

# For any outline that still has no parent, attach to book (defensive)
for oid in outline_ids:
    parents.setdefault(oid, book_eid)
    if parents[oid] == book_eid and oid not in children[book_eid]:
        children[book_eid].append(oid)

# stable order
label_map = dict(zip(df["id"], df["label"]))
for k in list(children.keys()):
    children[k] = sorted(set(children[k]), key=lambda nid: label_map.get(nid, ""))

# Fill df parent column
df["parent"] = df["id"].map(lambda nid: parents.get(nid, "") if nid != book_eid else "")
df.loc[df["id"] == book_eid, "parent"] = ""
dfi = df.set_index("id", drop=False)

# -----------------------------
# 4) Radial mind-map layout
# -----------------------------
def leaf_count(nid: str) -> int:
    kids = children.get(nid, [])
    if not kids:
        return 1
    return sum(leaf_count(k) for k in kids)

ids = set(df["id"])
leafs = {nid: leaf_count(nid) for nid in ids}

angle_center = {}
def assign_angles(nid: str, start: float, end: float):
    angle_center[nid] = (start + end) / 2.0
    kids = children.get(nid, [])
    if not kids:
        return
    total = sum(leafs[k] for k in kids) or len(kids)
    a = start
    for k in kids:
        w = (end - start) * (leafs[k] / total if total else 1/len(kids))
        assign_angles(k, a, a + w)
        a += w

assign_angles(book_eid, 0.0, 2.0 * math.pi)

# depths
depths = {book_eid: 0}
dq = deque([book_eid])
while dq:
    u = dq.popleft()
    for v in children.get(u, []):
        if v not in depths:
            depths[v] = depths[u] + 1
            dq.append(v)

# positions
pos = {}
for nid in ids:
    th = angle_center.get(nid, 0.0)
    r = depths.get(nid, 0) * RADIUS_STEP
    pos[nid] = (r * math.cos(th), r * math.sin(th))

# curved edges
def bezier(p0, p1, curvature=0.18, steps=10):
    (x0,y0),(x1,y1)=p0,p1
    mx,my=(x0+x1)/2.0,(y0+y1)/2.0
    cx,cy=mx*(1+curvature), my*(1+curvature)
    pts=[]
    for i in range(steps+1):
        t=i/steps
        x=(1-t)**2*x0+2*(1-t)*t*cx+t**2*x1
        y=(1-t)**2*y0+2*(1-t)*t*cy+t**2*y1
        pts.append((x,y))
    return pts

edge_x, edge_y = [], []
for child, par in parents.items():
    if child in pos and par in pos:
        pts = bezier(pos[par], pos[child])
        edge_x.extend([x for x,_ in pts] + [None])
        edge_y.extend([y for _,y in pts] + [None])

edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    mode="lines",
    line=dict(width=1),
    opacity=EDGE_ALPHA,
    hoverinfo="skip"
)

# nodes
node_x, node_y, node_text, node_hover, node_size, node_color = [], [], [], [], [], []
for nid in ids:
    x,y = pos[nid]
    t = dfi.loc[nid, "type"]
    node_x.append(x); node_y.append(y)

    d = depths.get(nid, 0)
    lbl = dfi.loc[nid, "label"]
    # reduce clutter: label only shallow levels + book
    if nid == book_eid or (t == "outline" and d <= LABEL_DEPTH):
        node_text.append(lbl)
    else:
        node_text.append("")

    node_hover.append(
        f"<b>{lbl}</b><br>"
        f"type: {t}<br>"
        f"depth: {d}<br>"
        f"pages: {dfi.loc[nid,'pages']}<br>"
        f"id: {nid}"
    )

    if nid == book_eid:
        node_size.append(30)
        node_color.append("rgba(30,30,30,0.95)")
    else:
        node_size.append(9)
        node_color.append("rgba(59,130,246,0.75)")

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode="markers+text",
    text=node_text,
    textposition="middle right",
    hovertext=node_hover,
    hoverinfo="text",
    marker=dict(size=node_size, color=node_color, line=dict(width=0.5, color="rgba(20,20,20,0.35)")),
    textfont=dict(size=13),
)

fig = go.Figure([edge_trace, node_trace])
fig.update_layout(
    title=f"REAL-E-CON — Mind-Map (Neo4j direct, outlines only) — {BOOK_ID}",
    showlegend=False,
    height=FIG_HEIGHT,
    margin=dict(l=10, r=10, t=60, b=10),
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
)
# keep aspect ratio so it stays circular
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.show()

Book: pindyck_micro_9e | title: Microeconomics
Reachable outlines: 468
HAS_CHILD edges (pruned): 451
Root outlines (HAS_OUTLINE): 17


In [19]:
LAYOUT_STYLE = "bach"   # "bach" or "radial"

# Bach-style layout knobs
BACH_SPREAD = 120        # horizontal spacing between depth levels
BACH_VSPACE = 24         # vertical spacing per leaf slot
BACH_BRANCH_GAP = 120    # vertical gap between main branches
BACH_CURVE = 0.35        # edge curvature
BACH_LABEL_MAXLEN = 70   # truncate labels

In [27]:
# ==========================================================
# Bach-style mind map (spaced + SVG export)
# ==========================================================
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path

# ---- spacing / readability knobs ----
BACH_SPREAD = 220          # horizontal spacing per depth (bigger = more room)
BACH_VSPACE = 34           # vertical spacing per leaf slot within a branch
BACH_BRANCH_GAP = 220      # vertical gap between main branches
BACH_CURVE = 0.35          # edge curvature
BACH_LABEL_MAXLEN = 72
LABEL_DEPTH = 3            # show labels only up to this depth (hover still works)
FONT_SIZE = 11

# Canvas sizing (big by default; SVG handles it well)
SVG_WIDTH = 3200
SVG_HEIGHT = 2000
SVG_SCALE = 2              # higher = crisper text/lines

# After initial layout, rescale to target y-span to reduce collisions
TARGET_Y_SPAN = 2400       # increase to spread vertically more (e.g., 3200)

OUT_SVG = Path(f"REAL_E_CON_BachMindMap_{BOOK_ID}.svg")

# ----------------------------------------------------------
# Helpers
# ----------------------------------------------------------
main = children.get(book_eid, [])
if not main:
    raise RuntimeError("Book has no children in children[]. Ensure Book->root outlines edges are included.")

def subtree_size(nid):
    stack=[nid]; seen=set(); n=0
    while stack:
        x=stack.pop()
        if x in seen:
            continue
        seen.add(x); n += 1
        stack.extend(children.get(x, []))
    return n

main = sorted(main, key=subtree_size, reverse=True)

# balance left/right by subtree size
left, right = [], []
ls = rs = 0
for m in main:
    s = subtree_size(m)
    if ls <= rs:
        left.append(m); ls += s
    else:
        right.append(m); rs += s

def collect_leaves(nid):
    out=[]
    stack=[nid]
    while stack:
        x=stack.pop()
        kids=children.get(x, [])
        if not kids:
            out.append(x)
        else:
            for k in reversed(kids):  # deterministic
                stack.append(k)
    return out

pos = {}
pos[book_eid] = (0.0, 0.0)

def assign_branch_positions(branch_root, side, y0):
    leaves = collect_leaves(branch_root)
    leaf_y = {leaf: y0 + i*BACH_VSPACE for i, leaf in enumerate(leaves)}

    # place leaves
    for leaf in leaves:
        d = depths.get(leaf, 0)
        x = (d * BACH_SPREAD) * (1 if side=="right" else -1)
        pos[leaf] = (x, leaf_y[leaf])

    # post-order to place internal nodes as mean(child y)
    stack=[branch_root]
    order=[]
    seen=set()
    while stack:
        x=stack.pop()
        if x in seen:
            continue
        seen.add(x)
        order.append(x)
        for k in children.get(x, []):
            stack.append(k)

    for x in sorted(order, key=lambda n: depths.get(n,0), reverse=True):
        kids = children.get(x, [])
        if not kids:
            continue
        ys = [pos[k][1] for k in kids if k in pos]
        if ys:
            y = float(np.mean(ys))
            d = depths.get(x, 0)
            xcoord = (d * BACH_SPREAD) * (1 if side=="right" else -1)
            pos[x] = (xcoord, y)

    if branch_root not in pos:
        d = depths.get(branch_root, 1)
        pos[branch_root] = ((d*BACH_SPREAD)*(1 if side=="right" else -1), y0)

    return (leaf_y[leaves[-1]] if leaves else y0)

# place left branches
y_cursor = - (len(left) * BACH_BRANCH_GAP)/2
for br in left:
    y_end = assign_branch_positions(br, "left", y_cursor)
    y_cursor = y_end + BACH_BRANCH_GAP

# place right branches
y_cursor = - (len(right) * BACH_BRANCH_GAP)/2
for br in right:
    y_end = assign_branch_positions(br, "right", y_cursor)
    y_cursor = y_end + BACH_BRANCH_GAP

# put main branch roots near center explicitly
for br in main:
    x, y = pos.get(br, (BACH_SPREAD, 0))
    side = "right" if br in right else "left"
    pos[br] = ((1*BACH_SPREAD)*(1 if side=="right" else -1), y)

# Rescale Y to target span (gives you predictable spacing and less overlap)
ys = [p[1] for p in pos.values()]
ymin, ymax = min(ys), max(ys)
span = max(1e-9, ymax - ymin)
scale_y = TARGET_Y_SPAN / span
for k in list(pos.keys()):
    x, y = pos[k]
    pos[k] = (x, y * scale_y)

# ----------------------------------------------------------
# Color per main branch (like the screenshot)
# ----------------------------------------------------------
def hsl(i, n):
    h = (i/max(1,n))*360.0
    return f"hsl({h:.0f},70%,45%)"

main_index = {m:i for i,m in enumerate(main)}

def branch_root_of(nid):
    cur = nid
    while True:
        p = parents.get(cur)
        if p == book_eid:
            return cur
        if not p:
            return None
        cur = p

def branch_color(nid):
    r = branch_root_of(nid)
    if r in main_index:
        return hsl(main_index[r], len(main))
    return "rgba(140,140,140,0.6)"

# ----------------------------------------------------------
# Edges (split per branch so each branch has its own color)
# ----------------------------------------------------------
def bezier_pts(p0, p1, side_sign=1, curvature=BACH_CURVE, steps=14):
    (x0,y0),(x1,y1)=p0,p1
    mx,my = (x0+x1)/2, (y0+y1)/2
    cx = mx + curvature * abs(x1-x0) * side_sign
    cy = my
    pts=[]
    for i in range(steps+1):
        t=i/steps
        x=(1-t)**2*x0 + 2*(1-t)*t*cx + t**2*x1
        y=(1-t)**2*y0 + 2*(1-t)*t*cy + t**2*y1
        pts.append((x,y))
    return pts

# group edges by branch root for coloring
edges_by_branch = {}
for child, par in parents.items():
    if child not in pos or par not in pos:
        continue
    # edge belongs to child's branch
    br = branch_root_of(child) or "__other__"
    edges_by_branch.setdefault(br, []).append((par, child))

edge_traces = []
for br, eds in edges_by_branch.items():
    ex, ey = [], []
    for par, child in eds:
        sx = 1 if pos[child][0] >= 0 else -1
        pts = bezier_pts(pos[par], pos[child], side_sign=sx)
        ex += [p[0] for p in pts] + [None]
        ey += [p[1] for p in pts] + [None]
    col = branch_color(br) if br != "__other__" else "rgba(160,160,160,0.45)"
    edge_traces.append(go.Scatter(
        x=ex, y=ey,
        mode="lines",
        line=dict(width=2, color=col),
        opacity=0.55,
        hoverinfo="skip"
    ))

# ----------------------------------------------------------
# Nodes + labels (two traces: left labels on left, right labels on right)
# ----------------------------------------------------------
def trunc(s, n=BACH_LABEL_MAXLEN):
    s = s or ""
    return (s[:n-1] + "…") if len(s) > n else s

left_x, left_y, left_text, left_hover, left_size, left_color, left_textpos = [],[],[],[],[],[],[]
right_x, right_y, right_text, right_hover, right_size, right_color, right_textpos = [],[],[],[],[],[],[]

for nid in ids:
    if nid not in pos:
        continue
    x, y = pos[nid]
    t = dfi.loc[nid, "type"]
    d = depths.get(nid, 0)
    lbl_full = dfi.loc[nid, "label"]
    lbl = trunc(lbl_full)

    show_label = (nid == book_eid) or (t == "outline" and d <= LABEL_DEPTH)

    hover = (
        f"<b>{lbl_full}</b><br>"
        f"type: {t}<br>"
        f"depth: {d}<br>"
        f"pages: {dfi.loc[nid,'pages']}<br>"
        f"id: {nid}"
    )

    if nid == book_eid:
        # keep book centered, label right
        right_x.append(x); right_y.append(y)
        right_text.append(lbl)
        right_hover.append(hover)
        right_size.append(18)
        right_color.append("rgba(20,20,20,0.95)")
        right_textpos.append("middle right")
        continue

    col = branch_color(nid)
    size = 6

    if x < 0:
        left_x.append(x); left_y.append(y)
        left_text.append(lbl if show_label else "")
        left_hover.append(hover)
        left_size.append(size)
        left_color.append(col)
        left_textpos.append("middle left")   # outside on left
    else:
        right_x.append(x); right_y.append(y)
        right_text.append(lbl if show_label else "")
        right_hover.append(hover)
        right_size.append(size)
        right_color.append(col)
        right_textpos.append("middle right") # outside on right

left_nodes = go.Scatter(
    x=left_x, y=left_y,
    mode="markers+text",
    text=left_text,
    textposition=left_textpos,
    hovertext=left_hover,
    hoverinfo="text",
    marker=dict(size=left_size, color=left_color, line=dict(width=0)),
    textfont=dict(size=FONT_SIZE, family="Arial"),
)

right_nodes = go.Scatter(
    x=right_x, y=right_y,
    mode="markers+text",
    text=right_text,
    textposition=right_textpos,
    hovertext=right_hover,
    hoverinfo="text",
    marker=dict(size=right_size, color=right_color, line=dict(width=0)),
    textfont=dict(size=FONT_SIZE, family="Arial"),
)

fig = go.Figure(edge_traces + [left_nodes, right_nodes])
fig.update_layout(
    title=f"REAL-E-CON — Bach-style Mind Map — {BOOK_ID}",
    showlegend=False,
    height=SVG_HEIGHT,  # display size in notebook (export uses width/height below)
    margin=dict(l=20, r=20, t=60, b=20),
    paper_bgcolor="white",
    plot_bgcolor="white",
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
)

fig.show()

# ----------------------------------------------------------
# HTML export (interactive, no kaleido needed)
# ----------------------------------------------------------
from pathlib import Path

OUT_HTML = Path(f"REAL_E_CON_BachMindMap_{BOOK_ID}.html")

# full_html=True makes it standalone (double-click to open)
fig.write_html(
    str(OUT_HTML),
    full_html=True,
    include_plotlyjs="cdn"  # use "embed" if you want a single-file with plotly bundled
)

print("Saved HTML:", OUT_HTML.resolve())


Saved HTML: /Users/pstaif/Downloads/MyApps/econ_visual/REAL_E_CON_BachMindMap_pindyck_micro_9e.html


In [50]:
# ==========================================================
# REAL-E-CON — Bach-style Mind Map (Neo4j → Plotly → HTML)
# - spaced out for readability
# - colors keyed to REAL numbered chapters (1.., 2.., ...)
# - front matter (Preface/Glossary/etc.) de-emphasized (grey)
# - exports interactive HTML
# ==========================================================

import os
import re
import math
from pathlib import Path
from collections import defaultdict, deque

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from neo4j import GraphDatabase

# -----------------------------
# CONFIG
# -----------------------------
BOOK_ID = "pindyck_micro_9e"

# Neo4j (expects these are defined earlier; will fallback to env vars)
NEO4J_URI  = globals().get("NEO4J_URI")  or os.getenv("NEO4J_URI", "neo4j://localhost:7687")
NEO4J_USER = globals().get("NEO4J_USER") or os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASS = (
    globals().get("NEO4J_PASS")
    or globals().get("NEO4J_PASSWORD")
    or os.getenv("NEO4J_PASS")
    or os.getenv("NEO4J_PASSWORD")
)
if not NEO4J_PASS:
    raise RuntimeError("Missing Neo4j password. Set NEO4J_PASS (or NEO4J_PASSWORD) in notebook or env.")

# Outline traversal
MAX_DEPTH = 30  # outline depth only; safe upper bound

# Bach-style spacing (increase further if you want MORE whitespace)
BACH_SPREAD     = 320   # horizontal spacing per depth
BACH_VSPACE     = 54    # vertical spacing per leaf slot within a branch
BACH_BRANCH_GAP = 520   # gap between main branches (book children)
BACH_CURVE      = 0.35  # edge curvature
LABEL_DEPTH     = 4     # show labels for book + chapters + outlines up to this depth
LEAF_VSPACE = int(BACH_VSPACE * 2.4)  # try 1.6–2.4

# Label control
LABEL_MAXLEN = 80

# HTML export
OUT_HTML = Path(f"REAL_E_CON_BachMindMap_{BOOK_ID}.html")
PLOTLY_JS = "embed"   # "embed" = fully offline single-file; "cdn" = smaller but needs internet

# Display size (in-notebook)
FIG_WIDTH  = 2200
FIG_HEIGHT = 1400

# -----------------------------
# 1) Pull book + reachable outlines + HAS_CHILD edges (pruned)
# -----------------------------
depth = int(MAX_DEPTH)
cypher = f"""
MATCH (b:Book {{book_id:$book_id}})
OPTIONAL MATCH (b)-[:HAS_OUTLINE]->(root:Outline)
OPTIONAL MATCH (root)-[:HAS_CHILD*0..{depth}]->(o:Outline)
WITH b, collect(DISTINCT root) AS roots, (collect(DISTINCT root) + collect(DISTINCT o)) AS outs
UNWIND outs AS n
WITH b, roots, collect(DISTINCT n) AS nodes
OPTIONAL MATCH (a:Outline)-[:HAS_CHILD]->(c:Outline)
WHERE a IN nodes AND c IN nodes
RETURN
  elementId(b) AS book_eid,
  coalesce(b.title,b.name,b.book_id) AS book_title,
  [x IN roots | elementId(x)] AS root_eids,
  [x IN nodes | {{
    eid: elementId(x),
    title: coalesce(x.title,x.name,x.outline_id),
    kind: coalesce(x.kind,'outline'),
    level: x.level,
    sp: x.start_page,
    ep: x.end_page
  }}] AS outline_nodes,
  collect({{src: elementId(a), dst: elementId(c)}}) AS child_edges
LIMIT 1
"""

drv = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
with drv.session() as sess:
    rec = sess.run(cypher, book_id=BOOK_ID).single()
drv.close()

if rec is None:
    raise RuntimeError("Neo4j returned no data. Check NEO4J_URI/auth/BOOK_ID.")

book_eid   = str(rec["book_eid"])
book_title = rec["book_title"]
root_eids  = [str(x) for x in (rec["root_eids"] or [])]
outline_nodes = rec["outline_nodes"] or []
child_edges   = rec["child_edges"] or []

if not outline_nodes:
    raise RuntimeError("No Outline nodes reachable. Check ingestion: Book-[:HAS_OUTLINE]->Outline and Outline-[:HAS_CHILD]->Outline.")

print("Book:", BOOK_ID, "| title:", book_title)
print("Reachable outlines:", len(outline_nodes))
print("HAS_CHILD edges (pruned):", len(child_edges))
print("Root outlines (HAS_OUTLINE):", len(root_eids))

# -----------------------------
# 2) Build node table + adjacency (include Book->Roots edges)
# -----------------------------
def fmt_pages(sp, ep):
    if sp is None and ep is None: return None
    if sp is None: return f"–{ep}"
    if ep is None: return f"{sp}–"
    return f"{sp}–{ep}"

rows = [{"id": book_eid, "parent": "", "label": book_title, "type": "book", "pages": None}]

for n in outline_nodes:
    nid = str(n["eid"])
    rows.append({
        "id": nid,
        "parent": None,
        "label": (n["title"] if n["title"] is not None else nid),
        "type": "outline",
        "pages": fmt_pages(n.get("sp"), n.get("ep"))
    })

df = pd.DataFrame(rows).drop_duplicates(subset=["id"]).reset_index(drop=True)
df["id"] = df["id"].astype(str)
dfi = df.set_index("id", drop=False)

outline_ids = set(df[df["type"] == "outline"]["id"])
label_map = dict(zip(df["id"], df["label"]))

children = defaultdict(list)
parents = {}

# Book -> roots (HAS_OUTLINE)
for r in root_eids:
    if r in outline_ids:
        parents.setdefault(r, book_eid)
        children[book_eid].append(r)

# Outline -> Outline (HAS_CHILD)
for e in child_edges:
    s = str(e["src"]); t = str(e["dst"])
    if s in outline_ids and t in outline_ids:
        children[s].append(t)
        parents.setdefault(t, s)

# Attach any orphan outline to book (defensive)
for oid in outline_ids:
    parents.setdefault(oid, book_eid)
    if parents[oid] == book_eid:
        children[book_eid].append(oid)

# Dedup + stable sort
for k in list(children.keys()):
    children[k] = sorted(set(children[k]), key=lambda nid: label_map.get(nid, ""))

# Fill df parent column
df["parent"] = df["id"].map(lambda nid: parents.get(nid, "") if nid != book_eid else "")
df.loc[df["id"] == book_eid, "parent"] = ""
dfi = df.set_index("id", drop=False)

ids = set(df["id"])

# Depths (for labeling + x placement)
depths = {book_eid: 0}
dq = deque([book_eid])
while dq:
    u = dq.popleft()
    for v in children.get(u, []):
        if v not in depths:
            depths[v] = depths[u] + 1
            dq.append(v)

# -----------------------------
# 3) Bach-style layout (left/right balanced by subtree size)
# -----------------------------
def subtree_size(nid):
    stack=[nid]; seen=set(); n=0
    while stack:
        x=stack.pop()
        if x in seen: 
            continue
        seen.add(x); n += 1
        stack.extend(children.get(x, []))
    return n

main = children.get(book_eid, [])
if not main:
    raise RuntimeError("Book has no children. Book->root edges missing in children[].")

main = sorted(main, key=subtree_size, reverse=True)

# balance left/right by subtree size
left, right = [], []
ls = rs = 0
for m in main:
    s = subtree_size(m)
    if ls <= rs:
        left.append(m); ls += s
    else:
        right.append(m); rs += s

def collect_leaves(nid):
    out=[]
    stack=[nid]
    while stack:
        x=stack.pop()
        kids=children.get(x, [])
        if not kids:
            out.append(x)
        else:
            for k in reversed(kids):
                stack.append(k)
    return out

pos = {book_eid: (0.0, 0.0)}

def assign_branch_positions(branch_root, side, y0):
    leaves = collect_leaves(branch_root)
    leaf_y = {leaf: y0 + i*LEAF_VSPACE for i, leaf in enumerate(leaves)}

    # place leaves
    for leaf in leaves:
        d = depths.get(leaf, 0)
        x = (d * BACH_SPREAD) * (1 if side=="right" else -1)
        pos[leaf] = (x, leaf_y[leaf])

    # post-order: internal nodes y = mean(children y)
    stack=[branch_root]
    order=[]
    seen=set()
    while stack:
        x=stack.pop()
        if x in seen:
            continue
        seen.add(x)
        order.append(x)
        for k in children.get(x, []):
            stack.append(k)

    for x in sorted(order, key=lambda n: depths.get(n,0), reverse=True):
        kids = children.get(x, [])
        if not kids:
            continue
        ys = [pos[k][1] for k in kids if k in pos]
        if ys:
            y = float(np.mean(ys))
            d = depths.get(x, 0)
            xcoord = (d * BACH_SPREAD) * (1 if side=="right" else -1)
            pos[x] = (xcoord, y)

    if branch_root not in pos:
        d = depths.get(branch_root, 1)
        pos[branch_root] = ((d*BACH_SPREAD)*(1 if side=="right" else -1), y0)

    return (leaf_y[leaves[-1]] if leaves else y0)

# Place branches with large gaps
y_cursor = - (len(left) * BACH_BRANCH_GAP)/2
for br in left:
    y_end = assign_branch_positions(br, "left", y_cursor)
    y_cursor = y_end + BACH_BRANCH_GAP

y_cursor = - (len(right) * BACH_BRANCH_GAP)/2
for br in right:
    y_end = assign_branch_positions(br, "right", y_cursor)
    y_cursor = y_end + BACH_BRANCH_GAP

# Put book's direct children near center at depth=1 x-position
for br in main:
    x, y = pos.get(br, (BACH_SPREAD, 0))
    side = "right" if br in right else "left"
    pos[br] = ((1*BACH_SPREAD)*(1 if side=="right" else -1), y)

# Dynamic Y rescale for readability
ys = [p[1] for p in pos.values()]
ymin, ymax = min(ys), max(ys)
span = max(1e-9, ymax - ymin)

total_leaves = sum(1 for nid in pos.keys() if not children.get(nid))
target_span = max(8200, int(total_leaves * (LEAF_VSPACE * 0.85)))

scale_y = target_span / span
for k in list(pos.keys()):
    x, y = pos[k]
    pos[k] = (x, y * scale_y)

# -----------------------------
# 4) Chapter-based coloring (numbers at start of outline title)
# -----------------------------
def chapter_num(label: str):
    if not label:
        return None
    m = re.match(r"^\s*(\d{1,2})\s+", label)  # "12 Oligopoly"
    if m:
        return int(m.group(1))
    m = re.match(r"^\s*chapter\s+(\d{1,2})\b", label, flags=re.I)  # "Chapter 12 ..."
    if m:
        return int(m.group(1))
    return None

# find numbered chapter anchors among outline nodes
chapter_anchors = []
for nid in outline_ids:
    num = chapter_num(label_map.get(nid, ""))
    if num is not None:
        chapter_anchors.append((num, nid))

chapter_anchors.sort(key=lambda x: x[0])

seen = set()
chapter_nodes = []
for num, nid in chapter_anchors:
    if num not in seen:
        seen.add(num)
        chapter_nodes.append(nid)

chapter_index = {nid: i for i, nid in enumerate(chapter_nodes)}

def hsl(i, n):
    h = (i / max(1, n)) * 360.0
    return f"hsl({h:.0f},70%,45%)"

def chapter_anchor_of(nid: str):
    cur = nid
    while cur and cur != book_eid:
        if cur in chapter_index:
            return cur
        cur = parents.get(cur)
    return None

def node_color(nid: str):
    anc = chapter_anchor_of(nid)
    if anc is not None:
        return hsl(chapter_index[anc], len(chapter_nodes))
    # front matter or non-numbered sections: de-emphasize
    return "rgba(160,160,160,0.25)"

# -----------------------------
# 5) Plot edges (colored by chapter anchor) + nodes (labels outside)
# -----------------------------
def trunc(s: str, n: int = LABEL_MAXLEN):
    s = s or ""
    return (s[:n-1] + "…") if len(s) > n else s

def bezier_pts(p0, p1, side_sign=1, curvature=BACH_CURVE, steps=14):
    (x0,y0),(x1,y1)=p0,p1
    mx,my = (x0+x1)/2, (y0+y1)/2
    cx = mx + curvature * abs(x1-x0) * side_sign
    cy = my
    pts=[]
    for i in range(steps+1):
        t=i/steps
        x=(1-t)**2*x0 + 2*(1-t)*t*cx + t**2*x1
        y=(1-t)**2*y0 + 2*(1-t)*t*cy + t**2*y1
        pts.append((x,y))
    return pts

# group edges by chapter anchor of CHILD (so whole chapter branch is consistent)
edges_by_group = defaultdict(list)
for child, par in parents.items():
    if child not in pos or par not in pos:
        continue
    g = chapter_anchor_of(child) or "__nonchapter__"
    edges_by_group[g].append((par, child))

edge_traces = []
for g, eds in edges_by_group.items():
    ex, ey = [], []
    for par, child in eds:
        sx = 1 if pos[child][0] >= 0 else -1
        pts = bezier_pts(pos[par], pos[child], side_sign=sx)
        ex += [p[0] for p in pts] + [None]
        ey += [p[1] for p in pts] + [None]
    col = "rgba(180,180,180,0.22)" if g == "__nonchapter__" else node_color(g)
    edge_traces.append(go.Scatter(
        x=ex, y=ey,
        mode="lines",
        line=dict(width=2, color=col),
        opacity=0.65,
        hoverinfo="skip"
    ))

# Nodes split into left/right so labels sit on the OUTSIDE
lx, ly, ltext, lhover, lsize, lcol, lpos = [], [], [], [], [], [], []
rx, ry, rtext, rhover, rsize, rcol, rpos = [], [], [], [], [], [], []

for nid in ids:
    if nid not in pos:
        continue
    x, y = pos[nid]
    t = dfi.loc[nid, "type"]
    d = depths.get(nid, 0)
    lbl_full = dfi.loc[nid, "label"]
    lbl = trunc(lbl_full)

    is_chapter = (nid in chapter_index)
    show_label = (nid == book_eid) or is_chapter or (t == "outline" and d <= LABEL_DEPTH)

    hover = (
        f"<b>{lbl_full}</b><br>"
        f"type: {t}<br>"
        f"depth: {d}<br>"
        f"pages: {dfi.loc[nid,'pages']}<br>"
        f"id: {nid}"
    )

    if nid == book_eid:
        # keep book on right trace for simplicity
        rx.append(x); ry.append(y)
        rtext.append(lbl)
        rhover.append(hover)
        rsize.append(18)
        rcol.append("rgba(20,20,20,0.95)")
        rpos.append("middle right")
        continue

    col = node_color(nid)
    size = 7 if is_chapter else 5

    if x < 0:
        lx.append(x); ly.append(y)
        ltext.append(lbl if show_label else "")
        lhover.append(hover)
        lsize.append(size)
        lcol.append(col)
        lpos.append("middle left")     # outside left
    else:
        rx.append(x); ry.append(y)
        rtext.append(lbl if show_label else "")
        rhover.append(hover)
        rsize.append(size)
        rcol.append(col)
        rpos.append("middle right")    # outside right

left_nodes = go.Scatter(
    x=lx, y=ly,
    mode="markers+text",
    text=ltext,
    textposition=lpos,
    hovertext=lhover,
    hoverinfo="text",
    marker=dict(size=lsize, color=lcol, line=dict(width=0)),
    textfont=dict(size=11, family="Arial"),
)

right_nodes = go.Scatter(
    x=rx, y=ry,
    mode="markers+text",
    text=rtext,
    textposition=rpos,
    hovertext=rhover,
    hoverinfo="text",
    marker=dict(size=rsize, color=rcol, line=dict(width=0)),
    textfont=dict(size=11, family="Arial"),
)

fig = go.Figure(edge_traces + [left_nodes, right_nodes])
fig.update_layout(
    title=f"REAL-E-CON — Bach-style Mind Map — {BOOK_ID}",
    showlegend=False,
    width=FIG_WIDTH,
    height=FIG_HEIGHT,
    margin=dict(l=20, r=20, t=60, b=20),
    paper_bgcolor="white",
    plot_bgcolor="white",
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
)

fig.show()

# -----------------------------
# 6) Export HTML
# -----------------------------
fig.write_html(
    str(OUT_HTML),
    full_html=True,
    include_plotlyjs=PLOTLY_JS
)

print("Saved HTML:", OUT_HTML.resolve())
print("Tip: if file is huge, set PLOTLY_JS='cdn' (smaller but requires internet).")

Book: pindyck_micro_9e | title: Microeconomics
Reachable outlines: 468
HAS_CHILD edges (pruned): 451
Root outlines (HAS_OUTLINE): 17


Saved HTML: /Users/pstaif/Downloads/MyApps/econ_visual/REAL_E_CON_BachMindMap_pindyck_micro_9e.html
Tip: if file is huge, set PLOTLY_JS='cdn' (smaller but requires internet).


In [54]:
# ==========================================================
# REAL-E-CON — D3 Bach-style Mind Map (Neo4j → JSON → HTML)
# - True SVG text background boxes (rect behind label)
# - Clean spacing controls via d3.tree nodeSize/separation
# - Pan/zoom
# - Chapter-based colors (numeric prefix "12 ..." etc.)
# ==========================================================

import os
import re
import json
from pathlib import Path
from collections import defaultdict, deque

from neo4j import GraphDatabase
from IPython.display import IFrame, display

# -----------------------------
# CONFIG
# -----------------------------
BOOK_ID = "pindyck_micro_9e"

NEO4J_URI  = globals().get("NEO4J_URI")  or os.getenv("NEO4J_URI", "neo4j://localhost:7687")
NEO4J_USER = globals().get("NEO4J_USER") or os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASS = (
    globals().get("NEO4J_PASS")
    or globals().get("NEO4J_PASSWORD")
    or os.getenv("NEO4J_PASS")
    or os.getenv("NEO4J_PASSWORD")
)
if not NEO4J_PASS:
    raise RuntimeError("Missing Neo4j password. Set NEO4J_PASS (or NEO4J_PASSWORD).")

# How deep to traverse Outline->HAS_CHILD. Set high (30) for full outline.
MAX_DEPTH = 30

# Output
OUT_HTML = Path(f"REAL_E_CON_D3MindMap_{BOOK_ID}.html").resolve()

# -----------------------------
# 1) Load hierarchy from Neo4j
# -----------------------------
depth = int(MAX_DEPTH)

# We avoid parameterizing the *range* because Neo4j doesn't allow params in *1..$d*
cypher = f"""
MATCH (b:Book {{book_id:$book_id}})
OPTIONAL MATCH (b)-[:HAS_OUTLINE]->(root:Outline)
OPTIONAL MATCH (root)-[:HAS_CHILD*0..{depth}]->(o:Outline)
WITH b, collect(DISTINCT root) AS roots, (collect(DISTINCT root) + collect(DISTINCT o)) AS outs
UNWIND outs AS n
WITH b, roots, collect(DISTINCT n) AS nodes
OPTIONAL MATCH (a:Outline)-[:HAS_CHILD]->(c:Outline)
WHERE a IN nodes AND c IN nodes
RETURN
  elementId(b) AS book_eid,
  coalesce(b.title,b.name,b.book_id) AS book_title,
  [x IN roots | elementId(x)] AS root_eids,
  [x IN nodes | {{
    eid: elementId(x),
    title: coalesce(x.title,x.name,x.outline_id),
    start_page: x.start_page,
    end_page: x.end_page
  }}] AS outline_nodes,
  collect({{src: elementId(a), dst: elementId(c)}}) AS child_edges
LIMIT 1
"""

drv = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
with drv.session() as sess:
    rec = sess.run(cypher, book_id=BOOK_ID).single()
drv.close()

if rec is None:
    raise RuntimeError("No data returned. Check BOOK_ID and Neo4j connection.")

book_eid   = str(rec["book_eid"])
book_title = rec["book_title"] or BOOK_ID
root_eids  = [str(x) for x in (rec["root_eids"] or [])]
outline_nodes = rec["outline_nodes"] or []
child_edges   = rec["child_edges"] or []

if not outline_nodes:
    raise RuntimeError("No Outline nodes found. Ensure Book-[:HAS_OUTLINE]->Outline exists.")

print("Book:", BOOK_ID, "| title:", book_title)
print("Outlines:", len(outline_nodes), "| HAS_CHILD edges:", len(child_edges), "| roots:", len(root_eids))

# -----------------------------
# 2) Build tree JSON (Book -> roots -> children)
# -----------------------------
title_by = {}
meta_by  = {}

for n in outline_nodes:
    eid = str(n["eid"])
    title_by[eid] = (n.get("title") or eid).strip()
    meta_by[eid] = {
        "start_page": n.get("start_page"),
        "end_page": n.get("end_page"),
    }

children = defaultdict(list)
parent = {}

# Book -> roots
for r in root_eids:
    children[book_eid].append(r)
    parent.setdefault(r, book_eid)

# Outline -> Outline
for e in child_edges:
    s = str(e["src"]); t = str(e["dst"])
    children[s].append(t)
    parent.setdefault(t, s)

# sort children for stable layout
for k in list(children.keys()):
    children[k] = sorted(set(children[k]), key=lambda nid: title_by.get(nid, ""))

# ensure roots exist; if not, fall back to outlines with no parent
if not children[book_eid]:
    orphans = [nid for nid in title_by.keys() if nid not in parent]
    children[book_eid] = sorted(orphans, key=lambda nid: title_by.get(nid, ""))

# build recursive tree dict
def build_node(nid: str):
    if nid == book_eid:
        label = book_title
    else:
        label = title_by.get(nid, nid)

    node = {
        "id": nid,
        "label": label,
        "meta": meta_by.get(nid, {}),
        "children": []
    }
    for ch in children.get(nid, []):
        node["children"].append(build_node(ch))
    return node

tree = build_node(book_eid)

# -----------------------------
# 3) Write D3 HTML (Bach-style split left/right, curved links, boxed labels)
# -----------------------------
data_json = json.dumps(tree)

html = f"""<!doctype html>
<html>
<head>
  <meta charset="utf-8"/>
  <title>REAL-E-CON — D3 Mind Map — {BOOK_ID}</title>
  <style>
    html, body {{ margin:0; padding:0; width:100%; height:100%; background:#fff; font-family: -apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Arial,sans-serif; }}
    #wrap {{ width:100vw; height:100vh; overflow:hidden; }}
    .title {{
      position: fixed; left: 18px; top: 14px; z-index: 10;
      font-size: 28px; font-weight: 700; color: #1f2a44; opacity: 0.95;
      letter-spacing: 0.2px;
      background: rgba(255,255,255,0.85); padding: 6px 10px; border-radius: 10px;
      box-shadow: 0 6px 22px rgba(0,0,0,0.08);
    }}
    .hint {{
      position: fixed; left: 18px; top: 62px; z-index: 10;
      font-size: 13px; color: rgba(31,42,68,0.75);
      background: rgba(255,255,255,0.85); padding: 6px 10px; border-radius: 10px;
      box-shadow: 0 6px 22px rgba(0,0,0,0.06);
    }}
    svg {{ width:100%; height:100%; }}
    .link {{ fill: none; stroke-width: 2.0; stroke-linecap: round; opacity: 0.55; }}
    .node-circle {{ stroke: rgba(0,0,0,0.08); stroke-width: 1; }}
    .label-rect {{
      fill: rgba(255,255,255,0.92);
      stroke: rgba(0,0,0,0.10);
      stroke-width: 1;
      rx: 4; ry: 4;
    }}
    .label-text {{ font-size: 13px; fill: rgba(15,23,42,0.95); }}
    .label-text.chapter {{ font-weight: 700; font-size: 14px; }}
  </style>
</head>
<body>
  <div class="title">REAL-E-CON — Bach-style Mind Map — {BOOK_ID}</div>
  <div class="hint">Scroll/trackpad to zoom, drag to pan. Labels have real SVG background boxes.</div>
  <div id="wrap"></div>

  <script src="https://cdn.jsdelivr.net/npm/d3@7"></script>
  <script>
    const data = {data_json};

    // -----------------------------
    // Layout knobs (tune these)
    // -----------------------------
    const H_SPREAD = 240;    // horizontal distance per depth (bigger => more left/right spacing)
    const V_SPREAD = 32;     // vertical distance per node row (bigger => more separation)
    const ROOT_GAP = 160;    // gap between left and right roots from center
    const CURVE = 0.35;      // link curvature
    const LABEL_PAD_X = 6;
    const LABEL_PAD_Y = 3;

    // chapter color palette by chapter number
    function chapterNum(label) {{
      if (!label) return null;
      let m = /^\\s*(\\d{{1,2}})\\s+/.exec(label);
      if (m) return +m[1];
      m = /^\\s*chapter\\s+(\\d{{1,2}})\\b/i.exec(label);
      if (m) return +m[1];
      return null;
    }}

    function chapterAnchor(d) {{
      // walk up until we find a node that looks like a numbered chapter
      let cur = d;
      while (cur) {{
        const n = chapterNum(cur.data.label);
        if (n !== null) return n;
        cur = cur.parent;
      }}
      return null;
    }}

    const color = d3.scaleOrdinal()
      .domain(d3.range(1, 40))
      .range(d3.range(1, 40).map(i => `hsl(${{(i*23)%360}},70%,50%)`));

    // Create SVG + zoom layer
    const wrap = d3.select("#wrap");
    const svg = wrap.append("svg");
    const gZoom = svg.append("g");

    const zoom = d3.zoom()
      .scaleExtent([0.15, 5])
      .on("zoom", (event) => {{
        gZoom.attr("transform", event.transform);
      }});
    svg.call(zoom);

    // Split book children into left/right for mindmap style
    const root = d3.hierarchy(data);
    const children = root.children || [];
    // simple balancing by subtree size
    children.forEach(c => c._size = c.descendants().length);
    children.sort((a,b) => b._size - a._size);

    let leftKids = [], rightKids = [];
    let ls=0, rs=0;
    for (const c of children) {{
      if (ls <= rs) {{ leftKids.push(c); ls += c._size; }}
      else {{ rightKids.push(c); rs += c._size; }}
    }}

    const leftRoot = d3.hierarchy({{...data, children: leftKids.map(k => k.data)}});
    const rightRoot = d3.hierarchy({{...data, children: rightKids.map(k => k.data)}});

    // Tree layout
    const tree = d3.tree()
      .nodeSize([V_SPREAD, H_SPREAD])
      .separation((a,b) => {{
        // more separation for leaves to reduce overlap
        const aLeaf = !a.children || a.children.length===0;
        const bLeaf = !b.children || b.children.length===0;
        return (aLeaf || bLeaf) ? 1.35 : 1.05;
      }});

    tree(leftRoot);
    tree(rightRoot);

    // Center both at origin; mirror left on x-axis
    // D3 tree uses x vertical, y horizontal
    leftRoot.each(d => {{ d.y = -d.y - ROOT_GAP; }});
    rightRoot.each(d => {{ d.y = d.y + ROOT_GAP; }});

    // Combine nodes/links (avoid duplicating the center root circle/label twice)
    const nodes = []
      .concat(leftRoot.descendants())
      .concat(rightRoot.descendants().slice(1)); // drop duplicate root

    const links = []
      .concat(leftRoot.links())
      .concat(rightRoot.links().slice(1)); // drop duplicate root link

    // Compute bounds for initial view
    const xs = nodes.map(d => d.x);
    const ys = nodes.map(d => d.y);
    const minX = d3.min(xs), maxX = d3.max(xs);
    const minY = d3.min(ys), maxY = d3.max(ys);

    const pad = 200;
    const vb = [minY - pad, minX - pad, (maxY-minY) + 2*pad, (maxX-minX) + 2*pad];
    svg.attr("viewBox", vb.join(" "));

    // Link path (cubic bezier)
    function linkPath(d) {{
      const x0 = d.source.y, y0 = d.source.x;
      const x1 = d.target.y, y1 = d.target.x;
      const dx = x1 - x0;
      const cx0 = x0 + dx * CURVE;
      const cx1 = x1 - dx * CURVE;
      return `M${{x0}},${{y0}} C${{cx0}},${{y0}} ${{cx1}},${{y1}} ${{x1}},${{y1}}`;
    }}

    // Draw links first
    const linkSel = gZoom.append("g")
      .attr("class","links")
      .selectAll("path")
      .data(links)
      .join("path")
      .attr("class","link")
      .attr("d", linkPath)
      .attr("stroke", d => {{
        const n = chapterAnchor(d.target);
        return n ? color(n) : "rgba(160,160,160,0.35)";
      }});

    // Node groups
    const nodeSel = gZoom.append("g")
      .attr("class","nodes")
      .selectAll("g")
      .data(nodes)
      .join("g")
      .attr("transform", d => `translate(${{d.y}},${{d.x}})`);

    // circles
    nodeSel.append("circle")
      .attr("class","node-circle")
      .attr("r", d => {{
        if (!d.parent) return 11;
        const n = chapterNum(d.data.label);
        if (n !== null) return 7;
        return 4;
      }})
      .attr("fill", d => {{
        if (!d.parent) return "rgba(20,20,20,0.92)";
        const n = chapterAnchor(d);
        return n ? color(n) : "rgba(160,160,160,0.35)";
      }});

    // label group (rect + text)
    const labelG = nodeSel.append("g")
      .attr("class","label")
      .attr("transform", d => {{
        // outside alignment
        const isLeft = d.y < 0;
        const dx = isLeft ? -10 : 10;
        return `translate(${{dx}},0)`;
      }});

    const text = labelG.append("text")
      .attr("class", d => {{
        const isChap = chapterNum(d.data.label) !== null;
        return isChap ? "label-text chapter" : "label-text";
      }})
      .attr("text-anchor", d => d.y < 0 ? "end" : "start")
      .attr("dominant-baseline","middle")
      .text(d => d.data.label);

    // rect behind text (needs bbox after text rendered)
    labelG.insert("rect", "text")
      .attr("class","label-rect")
      .each(function(d) {{
        const t = d3.select(this.parentNode).select("text").node();
        const bb = t.getBBox();
        d3.select(this)
          .attr("x", bb.x - LABEL_PAD_X)
          .attr("y", bb.y - LABEL_PAD_Y)
          .attr("width", bb.width + 2*LABEL_PAD_X)
          .attr("height", bb.height + 2*LABEL_PAD_Y);
      }});

    // Tooltip on hover (simple title attribute)
    nodeSel.append("title")
      .text(d => {{
        const m = d.data.meta || {{}};
        const sp = m.start_page, ep = m.end_page;
        const pages = (sp==null && ep==null) ? "" : ` (pages: ${{sp ?? ""}}–${{ep ?? ""}})`;
        return d.data.label + pages;
      }});

  </script>
</body>
</html>
"""

OUT_HTML.write_text(html, encoding="utf-8")
print("Saved:", str(OUT_HTML))

# Show inline
display(IFrame(src=str(OUT_HTML), width="100%", height=900))

Book: pindyck_micro_9e | title: Microeconomics
Outlines: 468 | HAS_CHILD edges: 451 | roots: 17
Saved: /Users/pstaif/Downloads/MyApps/econ_visual/REAL_E_CON_D3MindMap_pindyck_micro_9e.html


In [55]:
# ==========================================================
# REAL-E-CON — D3 Bach-style Mind Map (Neo4j → JSON → HTML)
# + EXPORT BUTTONS: Download SVG + High-res PNG
#
# Output:
#   REAL_E_CON_D3MindMap_pindyck_micro_9e.html
# Features:
# - True SVG label background boxes (rect behind label via getBBox)
# - Pan/zoom
# - Chapter-based colors (numeric prefix)
# - Built-in export buttons:
#     * Download SVG (vector)
#     * Download PNG (4×; change scale in JS)
# ==========================================================

import os
import json
from pathlib import Path
from collections import defaultdict

from neo4j import GraphDatabase
from IPython.display import IFrame, display

# -----------------------------
# CONFIG
# -----------------------------
BOOK_ID = "pindyck_micro_9e"

NEO4J_URI  = globals().get("NEO4J_URI")  or os.getenv("NEO4J_URI", "neo4j://localhost:7687")
NEO4J_USER = globals().get("NEO4J_USER") or os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASS = (
    globals().get("NEO4J_PASS")
    or globals().get("NEO4J_PASSWORD")
    or os.getenv("NEO4J_PASS")
    or os.getenv("NEO4J_PASSWORD")
)
if not NEO4J_PASS:
    raise RuntimeError("Missing Neo4j password. Set NEO4J_PASS (or NEO4J_PASSWORD).")

MAX_DEPTH = 30
OUT_HTML = Path(f"REAL_E_CON_D3MindMap_{BOOK_ID}.html").resolve()

# -----------------------------
# 1) Load hierarchy from Neo4j
# -----------------------------
depth = int(MAX_DEPTH)

cypher = f"""
MATCH (b:Book {{book_id:$book_id}})
OPTIONAL MATCH (b)-[:HAS_OUTLINE]->(root:Outline)
OPTIONAL MATCH (root)-[:HAS_CHILD*0..{depth}]->(o:Outline)
WITH b, collect(DISTINCT root) AS roots, (collect(DISTINCT root) + collect(DISTINCT o)) AS outs
UNWIND outs AS n
WITH b, roots, collect(DISTINCT n) AS nodes
OPTIONAL MATCH (a:Outline)-[:HAS_CHILD]->(c:Outline)
WHERE a IN nodes AND c IN nodes
RETURN
  elementId(b) AS book_eid,
  coalesce(b.title,b.name,b.book_id) AS book_title,
  [x IN roots | elementId(x)] AS root_eids,
  [x IN nodes | {{
    eid: elementId(x),
    title: coalesce(x.title,x.name,x.outline_id),
    start_page: x.start_page,
    end_page: x.end_page
  }}] AS outline_nodes,
  collect({{src: elementId(a), dst: elementId(c)}}) AS child_edges
LIMIT 1
"""

drv = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
with drv.session() as sess:
    rec = sess.run(cypher, book_id=BOOK_ID).single()
drv.close()

if rec is None:
    raise RuntimeError("No data returned. Check BOOK_ID and Neo4j connection.")

book_eid   = str(rec["book_eid"])
book_title = rec["book_title"] or BOOK_ID
root_eids  = [str(x) for x in (rec["root_eids"] or [])]
outline_nodes = rec["outline_nodes"] or []
child_edges   = rec["child_edges"] or []

if not outline_nodes:
    raise RuntimeError("No Outline nodes found. Ensure Book-[:HAS_OUTLINE]->Outline exists.")

print("Book:", BOOK_ID, "| title:", book_title)
print("Outlines:", len(outline_nodes), "| HAS_CHILD edges:", len(child_edges), "| roots:", len(root_eids))

# -----------------------------
# 2) Build tree JSON (Book -> roots -> children)
# -----------------------------
title_by = {}
meta_by  = {}
for n in outline_nodes:
    eid = str(n["eid"])
    title_by[eid] = (n.get("title") or eid).strip()
    meta_by[eid] = {"start_page": n.get("start_page"), "end_page": n.get("end_page")}

children = defaultdict(list)
parent = {}

# Book -> roots
for r in root_eids:
    children[book_eid].append(r)
    parent.setdefault(r, book_eid)

# Outline -> Outline
for e in child_edges:
    s = str(e["src"]); t = str(e["dst"])
    children[s].append(t)
    parent.setdefault(t, s)

# stable ordering
for k in list(children.keys()):
    children[k] = sorted(set(children[k]), key=lambda nid: title_by.get(nid, ""))

# if roots missing, attach orphans to book
if not children[book_eid]:
    orphans = [nid for nid in title_by.keys() if nid not in parent]
    children[book_eid] = sorted(orphans, key=lambda nid: title_by.get(nid, ""))

def build_node(nid: str):
    if nid == book_eid:
        label = book_title
        meta = {}
    else:
        label = title_by.get(nid, nid)
        meta = meta_by.get(nid, {})
    return {
        "id": nid,
        "label": label,
        "meta": meta,
        "children": [build_node(ch) for ch in children.get(nid, [])]
    }

tree = build_node(book_eid)
data_json = json.dumps(tree)

# -----------------------------
# 3) Write D3 HTML with export buttons
# -----------------------------
html = f"""<!doctype html>
<html>
<head>
  <meta charset="utf-8"/>
  <title>REAL-E-CON — D3 Mind Map — {BOOK_ID}</title>
  <style>
    html, body {{ margin:0; padding:0; width:100%; height:100%; background:#fff;
      font-family: -apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Arial,sans-serif; }}
    #wrap {{ width:100vw; height:100vh; overflow:hidden; }}

    .title {{
      position: fixed; left: 18px; top: 14px; z-index: 10;
      font-size: 28px; font-weight: 700; color: #1f2a44; opacity: 0.95;
      letter-spacing: 0.2px;
      background: rgba(255,255,255,0.85); padding: 6px 10px; border-radius: 10px;
      box-shadow: 0 6px 22px rgba(0,0,0,0.08);
    }}
    .hint {{
      position: fixed; left: 18px; top: 62px; z-index: 10;
      font-size: 13px; color: rgba(31,42,68,0.75);
      background: rgba(255,255,255,0.85); padding: 6px 10px; border-radius: 10px;
      box-shadow: 0 6px 22px rgba(0,0,0,0.06);
    }}

    .tools {{
      position: fixed; right: 18px; top: 14px; z-index: 10;
      display: flex; gap: 10px;
    }}
    .tools button {{
      border: 0; cursor: pointer;
      font-size: 13px; font-weight: 600;
      color: rgba(31,42,68,0.92);
      background: rgba(255,255,255,0.90);
      padding: 8px 10px; border-radius: 10px;
      box-shadow: 0 6px 22px rgba(0,0,0,0.08);
    }}
    .tools button:hover {{ background: rgba(255,255,255,0.98); }}

    svg {{ width:100%; height:100%; background:#fff; }}
    .link {{ fill: none; stroke-width: 2.0; stroke-linecap: round; opacity: 0.55; }}
    .node-circle {{ stroke: rgba(0,0,0,0.08); stroke-width: 1; }}
    .label-rect {{
      fill: rgba(255,255,255,0.92);
      stroke: rgba(0,0,0,0.10);
      stroke-width: 1;
      rx: 4; ry: 4;
    }}
    .label-text {{ font-size: 13px; fill: rgba(15,23,42,0.95); }}
    .label-text.chapter {{ font-weight: 700; font-size: 14px; }}
  </style>
</head>
<body>
  <div class="title">REAL-E-CON — Bach-style Mind Map — {BOOK_ID}</div>
  <div class="hint">Scroll/trackpad to zoom, drag to pan. Use buttons to export SVG/PNG.</div>
  <div class="tools">
    <button id="dl_svg">Download SVG</button>
    <button id="dl_png">Download PNG (4×)</button>
  </div>
  <div id="wrap"></div>

  <script src="https://cdn.jsdelivr.net/npm/d3@7"></script>
  <script>
    const data = {data_json};

    // -----------------------------
    // Layout knobs (tune these)
    // -----------------------------
    const H_SPREAD = 240;    // horizontal distance per depth
    const V_SPREAD = 32;     // vertical distance per node row
    const ROOT_GAP = 160;    // gap between left and right roots
    const CURVE = 0.35;      // link curvature
    const LABEL_PAD_X = 6;
    const LABEL_PAD_Y = 3;

    function chapterNum(label) {{
      if (!label) return null;
      let m = /^\\s*(\\d{{1,2}})\\s+/.exec(label);
      if (m) return +m[1];
      m = /^\\s*chapter\\s+(\\d{{1,2}})\\b/i.exec(label);
      if (m) return +m[1];
      return null;
    }}

    function chapterAnchor(d) {{
      let cur = d;
      while (cur) {{
        const n = chapterNum(cur.data.label);
        if (n !== null) return n;
        cur = cur.parent;
      }}
      return null;
    }}

    const color = d3.scaleOrdinal()
      .domain(d3.range(1, 40))
      .range(d3.range(1, 40).map(i => `hsl(${{(i*23)%360}},70%,50%)`));

    // Create SVG + zoom layer
    const wrap = d3.select("#wrap");
    const svg = wrap.append("svg");
    const gZoom = svg.append("g");

    const zoom = d3.zoom()
      .scaleExtent([0.15, 6])
      .on("zoom", (event) => {{
        gZoom.attr("transform", event.transform);
      }});
    svg.call(zoom);

    // Split children into left/right mindmap halves
    const root = d3.hierarchy(data);
    const kids = root.children || [];
    kids.forEach(c => c._size = c.descendants().length);
    kids.sort((a,b) => b._size - a._size);

    let leftKids = [], rightKids = [];
    let ls=0, rs=0;
    for (const c of kids) {{
      if (ls <= rs) {{ leftKids.push(c); ls += c._size; }}
      else {{ rightKids.push(c); rs += c._size; }}
    }}

    const leftRoot = d3.hierarchy({{...data, children: leftKids.map(k => k.data)}});
    const rightRoot = d3.hierarchy({{...data, children: rightKids.map(k => k.data)}});

    // Tree layout
    const tree = d3.tree()
      .nodeSize([V_SPREAD, H_SPREAD])
      .separation((a,b) => {{
        const aLeaf = !a.children || a.children.length===0;
        const bLeaf = !b.children || b.children.length===0;
        return (aLeaf || bLeaf) ? 1.35 : 1.05;
      }});

    tree(leftRoot);
    tree(rightRoot);

    // Mirror left on horizontal axis
    leftRoot.each(d => {{ d.y = -d.y - ROOT_GAP; }});
    rightRoot.each(d => {{ d.y = d.y + ROOT_GAP; }});

    // Combine nodes/links (avoid duplicating center root)
    const nodes = leftRoot.descendants().concat(rightRoot.descendants().slice(1));
    const links = leftRoot.links().concat(rightRoot.links().slice(1));

    // Bounds for viewBox
    const xs = nodes.map(d => d.x);
    const ys = nodes.map(d => d.y);
    const minX = d3.min(xs), maxX = d3.max(xs);
    const minY = d3.min(ys), maxY = d3.max(ys);
    const pad = 220;
    const vb = [minY - pad, minX - pad, (maxY-minY) + 2*pad, (maxX-minX) + 2*pad];
    svg.attr("viewBox", vb.join(" "));

    // Link path
    function linkPath(d) {{
      const x0 = d.source.y, y0 = d.source.x;
      const x1 = d.target.y, y1 = d.target.x;
      const dx = x1 - x0;
      const cx0 = x0 + dx * CURVE;
      const cx1 = x1 - dx * CURVE;
      return `M${{x0}},${{y0}} C${{cx0}},${{y0}} ${{cx1}},${{y1}} ${{x1}},${{y1}}`;
    }}

    // Links
    gZoom.append("g")
      .attr("class","links")
      .selectAll("path")
      .data(links)
      .join("path")
      .attr("class","link")
      .attr("d", linkPath)
      .attr("stroke", d => {{
        const n = chapterAnchor(d.target);
        return n ? color(n) : "rgba(160,160,160,0.35)";
      }});

    // Nodes
    const nodeSel = gZoom.append("g")
      .attr("class","nodes")
      .selectAll("g")
      .data(nodes)
      .join("g")
      .attr("transform", d => `translate(${{d.y}},${{d.x}})`);

    nodeSel.append("circle")
      .attr("class","node-circle")
      .attr("r", d => {{
        if (!d.parent) return 11;
        const n = chapterNum(d.data.label);
        if (n !== null) return 7;
        return 4;
      }})
      .attr("fill", d => {{
        if (!d.parent) return "rgba(20,20,20,0.92)";
        const n = chapterAnchor(d);
        return n ? color(n) : "rgba(160,160,160,0.35)";
      }});

    const labelG = nodeSel.append("g")
      .attr("class","label")
      .attr("transform", d => {{
        const isLeft = d.y < 0;
        const dx = isLeft ? -10 : 10;
        return `translate(${{dx}},0)`;
      }});

    const text = labelG.append("text")
      .attr("class", d => {{
        const isChap = chapterNum(d.data.label) !== null;
        return isChap ? "label-text chapter" : "label-text";
      }})
      .attr("text-anchor", d => d.y < 0 ? "end" : "start")
      .attr("dominant-baseline","middle")
      .text(d => d.data.label);

    labelG.insert("rect", "text")
      .attr("class","label-rect")
      .each(function(d) {{
        const t = d3.select(this.parentNode).select("text").node();
        const bb = t.getBBox();
        d3.select(this)
          .attr("x", bb.x - LABEL_PAD_X)
          .attr("y", bb.y - LABEL_PAD_Y)
          .attr("width", bb.width + 2*LABEL_PAD_X)
          .attr("height", bb.height + 2*LABEL_PAD_Y);
      }});

    nodeSel.append("title")
      .text(d => {{
        const m = d.data.meta || {{}};
        const sp = m.start_page, ep = m.end_page;
        const pages = (sp==null && ep==null) ? "" : ` (pages: ${{sp ?? ""}}–${{ep ?? ""}})`;
        return d.data.label + pages;
      }});

    // -----------------------------
    // EXPORT: SVG + High-res PNG
    // -----------------------------
    function downloadBlob(blob, filename){{
      const a = document.createElement("a");
      a.href = URL.createObjectURL(blob);
      a.download = filename;
      document.body.appendChild(a);
      a.click();
      a.remove();
      setTimeout(() => URL.revokeObjectURL(a.href), 2000);
    }}

    function serializeSvg(svgEl){{
      if (!svgEl.getAttribute("viewBox")) {{
        const w = svgEl.clientWidth || 1200;
        const h = svgEl.clientHeight || 800;
        svgEl.setAttribute("viewBox", `0 0 ${{w}} ${{h}}`);
      }}
      const clone = svgEl.cloneNode(true);

      // inline white background for export
      const style = document.createElement("style");
      style.textContent = `svg {{ background: white; }}`;
      clone.insertBefore(style, clone.firstChild);

      const serializer = new XMLSerializer();
      let src = serializer.serializeToString(clone);

      // fix xmlns
      if (!src.match(/^<svg[^>]+xmlns=/)) {{
        src = src.replace(/^<svg/, '<svg xmlns="http://www.w3.org/2000/svg"');
      }}
      if (!src.match(/^<svg[^>]+xmlns:xlink=/)) {{
        src = src.replace(/^<svg/, '<svg xmlns:xlink="http://www.w3.org/1999/xlink"');
      }}
      return src;
    }}

    document.getElementById("dl_svg").addEventListener("click", () => {{
      const svgEl = document.querySelector("svg");
      const src = serializeSvg(svgEl);
      const blob = new Blob([src], {{type: "image/svg+xml;charset=utf-8"}});
      downloadBlob(blob, "REAL_E_CON_mindmap.svg");
    }});

    document.getElementById("dl_png").addEventListener("click", async () => {{
      const svgEl = document.querySelector("svg");
      const src = serializeSvg(svgEl);

      const vb = svgEl.viewBox.baseVal;
      const w = (vb && vb.width) ? vb.width : (svgEl.clientWidth || 1600);
      const h = (vb && vb.height) ? vb.height : (svgEl.clientHeight || 900);

      const scale = 4; // change to 6 for ultra high res
      const canvas = document.createElement("canvas");
      canvas.width = Math.round(w * scale);
      canvas.height = Math.round(h * scale);
      const ctx = canvas.getContext("2d");

      // white background
      ctx.fillStyle = "#ffffff";
      ctx.fillRect(0, 0, canvas.width, canvas.height);

      const img = new Image();
      const svgBlob = new Blob([src], {{type: "image/svg+xml;charset=utf-8"}});
      const url = URL.createObjectURL(svgBlob);

      await new Promise((resolve, reject) => {{
        img.onload = resolve;
        img.onerror = reject;
        img.src = url;
      }});
      URL.revokeObjectURL(url);

      ctx.setTransform(scale, 0, 0, scale, 0, 0);
      ctx.drawImage(img, 0, 0);

      canvas.toBlob((blob) => {{
        downloadBlob(blob, `REAL_E_CON_mindmap_${{scale}}x.png`);
      }}, "image/png");
    }});

  </script>
</body>
</html>
"""

OUT_HTML.write_text(html, encoding="utf-8")
print("Saved:", str(OUT_HTML))

# Show inline (in Jupyter/VSCode notebook)
display(IFrame(src=str(OUT_HTML), width="100%", height=900))

Book: pindyck_micro_9e | title: Microeconomics
Outlines: 468 | HAS_CHILD edges: 451 | roots: 17
Saved: /Users/pstaif/Downloads/MyApps/econ_visual/REAL_E_CON_D3MindMap_pindyck_micro_9e.html
